# Notebook 09 — Metric–Human Alignment

This notebook measures how well each automatic MT evaluation metric correlates with human MQM judgements across all five Indic languages, in both native-script and romanised conditions.

**Metrics evaluated:** BLEU, chrF, TER, BLEURT, BERTScore, COMET — computed under native script and romanised input.

**Correlation measures used:**
- **Spearman ρ** — rank-based, robust to outliers and non-linear relationships.
- **Pearson r** — linear correlation; reported alongside Spearman as a consistency check.

**Input:** per-language CSVs from `../../data/processed/` (produced by `03_metric_scoring.ipynb`).  
**Output:** long-form and pivot CSV tables saved to `../../results/tables/`.

---

**Prerequisite:** Run notebooks `01` through `03` before this one.

**Install dependencies if needed:**
```
pip install pandas numpy scipy
```

## Setup

Import libraries, define file paths, and set the column name constants that match the CSVs produced by `03_metric_scoring.ipynb`.

The `METRIC_COLS_NAT` and `METRIC_COLS_ROM` dictionaries map a short display label (used in the output table) to the actual column name written by the scoring notebook. This single mapping makes it straightforward to rename or add metrics without touching the analysis cells downstream.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../../data/processed')
OUT_DIR  = Path('../../results/tables')
OUT_DIR.mkdir(parents=True, exist_ok=True)

LANGUAGES = ['gujarati', 'tamil', 'malayalam', 'marathi', 'hindi']
ISO       = {'gujarati': 'GUJ', 'tamil': 'TAM', 'malayalam': 'MAL',
             'marathi': 'MAR', 'hindi': 'HIN'}

COL_HYP     = 'Translation'
COL_REF     = 'Reference'
COL_SRC     = 'Source'
COL_HYP_ROM = 'Translation_Transliteration_romanized'
COL_REF_ROM = 'Reference_Transliteration_romanized'
COL_MQM     = 'Human_scores'

# Native-script metric columns
METRIC_COLS_NAT = {
    'BLEU_nat':      'bleu_native',
    'chrF_nat':      'chrf_native',
    'TER_nat':       'ter_native',
    'BLEURT_nat':    'bleurt_native',
    'BERTScore_nat': 'bertscore_native',
    'COMET_nat':     'comet_native',
}

# Romanised metric columns
METRIC_COLS_ROM = {
    'BLEU_rom':      'bleu_romanised',
    'chrF_rom':      'chrf_romanised',
    'TER_rom':       'ter_romanised',
    'BLEURT_rom':    'bleurt_romanised',
    'BERTScore_rom': 'bertscore_romanised',
    'COMET_rom':     'comet_romanised',
}

print('Config loaded. DATA_DIR =', DATA_DIR)

## Loading the Language Files

Read one CSV per language. Each file contains the original translation data plus the metric scores and TP/IP columns added by earlier notebooks. All five DataFrames are collected into a `dfs` dictionary keyed by language name.

If a file is missing, the notebook prints an informative message and the final `assert` prevents silent partial runs.

In [ ]:
dfs = {}
for lang in LANGUAGES:
    fp = DATA_DIR / f'{lang}_indicmt.csv'
    if fp.exists():
        dfs[lang] = pd.read_csv(fp)
        print(f'{ISO[lang]}: {len(dfs[lang])} rows  |  columns: {list(dfs[lang].columns)}')
    else:
        print(f'MISSING: {fp} — run 03_metric_scoring.ipynb first')

assert len(dfs) == 5, 'One or more language files missing — run 03_metric_scoring.ipynb'

## Correlation Helper

The `corr_row` function computes Spearman ρ and Pearson r for a single (metric, language) pair against MQM human scores. Key design choices:

- **NaN-safe:** any row where either the metric or the MQM value is missing is dropped before computing, so partial data does not corrupt the result.
- **Minimum sample guard:** if fewer than 10 valid pairs remain after dropping, both coefficients are returned as `NaN` — too small a sample for meaningful rank correlation.
- **P-values included:** both the coefficient and its two-sided p-value are returned, allowing significance markers in the final table.

In [ ]:
def corr_row(series_metric: pd.Series, series_human: pd.Series,
             metric_name: str, lang: str) -> dict:
    """Compute Spearman ρ and Pearson r against MQM. Returns a result dict."""
    mask = series_metric.notna() & series_human.notna()
    x = series_metric[mask].values
    y = series_human[mask].values
    n = int(mask.sum())
    if n < 10:
        return dict(lang=lang, metric=metric_name, n=n,
                    spearman=np.nan, sp_pval=np.nan,
                    pearson=np.nan,  pe_pval=np.nan)
    sp_r, sp_p = stats.spearmanr(x, y)
    pe_r, pe_p = stats.pearsonr(x, y)
    return dict(lang=lang, metric=metric_name, n=n,
                spearman=round(sp_r, 3), sp_pval=sp_p,
                pearson=round(pe_r, 3),  pe_pval=pe_p)

print('Helper defined.')

## Computing Correlations

Iterate over all five languages and all twelve metric columns (six native-script, six romanised). For each combination, `corr_row` is called and the result is appended to `rows`.

Missing metric columns are reported as warnings rather than raising errors — the notebook continues cleanly even if a particular metric was not computed for a given language.

In [ ]:
rows = []
for lang in LANGUAGES:
    df  = dfs[lang]
    iso = ISO[lang]
    if COL_MQM not in df.columns:
        print(f'WARNING: {COL_MQM} not in {lang} — skipping')
        continue
    human = df[COL_MQM]

    for label, col in METRIC_COLS_NAT.items():
        if col in df.columns:
            rows.append(corr_row(df[col], human, label, iso))
        else:
            print(f'  MISSING column: {col} in {lang}')

    for label, col in METRIC_COLS_ROM.items():
        if col in df.columns:
            rows.append(corr_row(df[col], human, label, iso))
        else:
            print(f'  MISSING column: {col} in {lang}')

corr_df = pd.DataFrame(rows)
print(f'Computed {len(corr_df)} correlation entries.')
corr_df.head(14)

## Building the Alignment Table

Pivot the long-form results into a table with one row per metric and one column per language. Each cell shows:

```
ρ* (r*)   —   Spearman ρ (Pearson r),  * = p < 0.05
```

The table is split into two blocks — **native-script** metrics first, then **romanised** — so the effect of romanisation on each metric is immediately visible by comparing the two blocks row by row.

Note: TER is negatively oriented (lower edit distance = better translation quality), so its raw correlation sign with MQM will differ from the other metrics.

In [ ]:
METRIC_ORDER = [
    'BLEU_nat',   'chrF_nat',   'TER_nat',
    'BLEURT_nat', 'BERTScore_nat', 'COMET_nat',
    'BLEU_rom',   'chrF_rom',   'TER_rom',
    'BLEURT_rom', 'BERTScore_rom', 'COMET_rom',
]
LANG_ORDER = ['GUJ', 'TAM', 'MAL', 'MAR', 'HIN']

def sig_mark(pval):
    if pd.isna(pval):
        return ''
    return '*' if pval < 0.05 else ''

sp_pivot   = corr_df.pivot(index='metric', columns='lang', values='spearman')
pe_pivot   = corr_df.pivot(index='metric', columns='lang', values='pearson')
sp_p_pivot = corr_df.pivot(index='metric', columns='lang', values='sp_pval')
pe_p_pivot = corr_df.pivot(index='metric', columns='lang', values='pe_pval')

sp_pivot   = sp_pivot.reindex(index=METRIC_ORDER, columns=LANG_ORDER)
pe_pivot   = pe_pivot.reindex(index=METRIC_ORDER, columns=LANG_ORDER)
sp_p_pivot = sp_p_pivot.reindex(index=METRIC_ORDER, columns=LANG_ORDER)
pe_p_pivot = pe_p_pivot.reindex(index=METRIC_ORDER, columns=LANG_ORDER)

table_cells = {}
for lang in LANG_ORDER:
    col_vals = []
    for met in METRIC_ORDER:
        sp  = sp_pivot.loc[met, lang]
        pe  = pe_pivot.loc[met, lang]
        spm = sig_mark(sp_p_pivot.loc[met, lang])
        pem = sig_mark(pe_p_pivot.loc[met, lang])
        if pd.isna(sp) and pd.isna(pe):
            col_vals.append('—')
        else:
            col_vals.append(f'{sp:.3f}{spm} ({pe:.3f}{pem})')
    table_cells[lang] = col_vals

alignment_table = pd.DataFrame(table_cells, index=METRIC_ORDER)
alignment_table.index.name = 'Metric'

print('=== Metric–Human Alignment  |  Spearman ρ (Pearson r) vs. Human_scores ===')
print('* = p < 0.05.  TER: negatively oriented (lower = better).')
print()
native_block  = [m for m in METRIC_ORDER if m.endswith('_nat')]
roman_block   = [m for m in METRIC_ORDER if m.endswith('_rom')]
for block, label in [(native_block, 'Native script'), (roman_block, 'Romanised')]:
    print(f'--- {label} ---')
    print(f'{"Metric":<22}', '  '.join(f'{l:>18}' for l in LANG_ORDER))
    print('-' * 115)
    for met in block:
        vals = '  '.join(f'{str(alignment_table.loc[met, l]):>18}' for l in LANG_ORDER)
        print(f'{met:<22} {vals}')
    print()

## Native vs. Romanised Alignment Shift

For each metric and language, compute the percentage change in Spearman ρ when switching from native-script to romanised input:

$$\Delta\rho\% = \frac{\rho_{\text{rom}} - \rho_{\text{nat}}}{|\rho_{\text{nat}}|} \times 100$$

Neural encoder metrics (COMET, BLEURT) rely on subword representations from XLM-R or BERT-based encoders. Romanisation changes the surface form of the text, disrupting the subword vocabulary these models were pretrained on, which is reflected in a large drop in their correlation with human scores.

Surface metrics (BLEU, chrF, TER) operate on character or n-gram overlap directly, so their alignment with human judgements is largely unaffected by the romanisation step.

In [ ]:
print('Spearman ρ shift under romanisation (Δρ%):')
print(f'{"Lang":>5}  {"COMET":>10}  {"BLEURT":>10}  {"BLEU":>8}  {"chrF":>8}  {"TER":>8}')
print('-' * 60)

collapse_rows = []
for lang in LANG_ORDER:
    row = {'lang': lang}
    for m_base in ['COMET', 'BLEURT', 'BLEU', 'chrF', 'TER']:
        nat = sp_pivot.loc[f'{m_base}_nat', lang]
        rom = sp_pivot.loc[f'{m_base}_rom', lang]
        if pd.notna(nat) and pd.notna(rom) and nat != 0:
            pct = round((rom - nat) / abs(nat) * 100, 1)
        else:
            pct = np.nan
        row[m_base] = pct
    collapse_rows.append(row)
    print(f'{lang:>5}  {row["COMET"]:>10}  {row["BLEURT"]:>10}  '
          f'{row["BLEU"]:>8}  {row["chrF"]:>8}  {row["TER"]:>8}')

collapse_df = pd.DataFrame(collapse_rows).set_index('lang')
print()
print('Neural encoder metrics (COMET, BLEURT): large negative Δρ% under romanisation.')
print('Surface metrics (BLEU, chrF, TER): Δρ% expected near zero.')

## Saving Results

Two files are written to `../../results/tables/`:

1. **`metric_human_alignment_long.csv`** — long-form table with one row per (language, metric) pair, containing Spearman ρ, Pearson r, sample size, and p-values. Suitable for downstream statistical analysis or LaTeX generation.
2. **`metric_human_alignment_pivot.csv`** — wide pivot in the display format shown above, with formatted `ρ (r)` cells.

In [ ]:
long_path  = OUT_DIR / 'metric_human_alignment_long.csv'
pivot_path = OUT_DIR / 'metric_human_alignment_pivot.csv'

corr_df[['lang', 'metric', 'n', 'spearman', 'sp_pval', 'pearson', 'pe_pval']].to_csv(
    long_path, index=False
)
alignment_table.to_csv(pivot_path)

print(f'Saved: {long_path}')
print(f'Saved: {pivot_path}')
print('\nDone — metric–human alignment analysis complete.')

## References

**Human MQM evaluation framework:**  
Freitag, M., Foster, G., Grangier, D., Ratnakar, V., Tan, Q., & Macherey, W. (2021). Experts, Errors, and Context: A Large-Scale Study of Human Evaluation for Machine Translation. *TACL*, 9, 1460–1474. https://aclanthology.org/2021.tacl-1.87

**COMET (neural MT metric):**  
Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural Framework for MT Evaluation. *EMNLP 2020*, pp. 2685–2702. https://aclanthology.org/2020.emnlp-main.213

**BLEURT:**  
Sellam, T., Das, D., & Parikh, A. P. (2020). BLEURT: Learning Robust Metrics for Text Generation. *ACL 2020*, pp. 7881–7892. https://aclanthology.org/2020.acl-main.704

**BERTScore:**  
Zhang, T., Kishore, V., Wu, F., Weinberger, K. Q., & Artzi, Y. (2020). BERTScore: Evaluating Text Generation with BERT. *ICLR 2020*. https://arxiv.org/abs/1904.09675

**IndicMT Eval dataset:**  
Sai B., A., Dixit, T., Nagarajan, V., Kunchukuttan, A., Kumar, P., Khapra, M. M., & Dabre, R. (2023). IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics for Indian Languages. *ACL 2023*, pp. 14210–14228. https://aclanthology.org/2023.acl-long.795

**Tokenization fairness across languages:**  
Petrov, A., La Malfa, E., Torr, P. H. S., & Bibi, A. (2023). Language Model Tokenizers Introduce Unfairness Between Languages. *NeurIPS 36*. https://arxiv.org/abs/2305.15425

**XLM-RoBERTa (tokenizer backbone):**  
Conneau, A., Khandelwal, K., Goyal, N., Chaudhary, V., Wenzek, G., Guzmán, F., Grave, E., Ott, M., Zettlemoyer, L., & Stoyanov, V. (2020). Unsupervised Cross-lingual Representation Learning at Scale. *ACL 2020*, pp. 8440–8451. https://arxiv.org/abs/1911.02116